In [ ]:
# --- Colab bootstrap: in Colab, clone + install the repo; locally, do nothing ---
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/CKmellow/per3-genotype-clock-gene-timing.git"
BRANCH = "main"  # change to your branch name to test it in Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content/per3-genotype-clock-gene-timing")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    src = str(repo / "src")
else:  # local: find the repo root (works from notebooks/ or notebooks/stages/)
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)
    src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)  # makes `import per3clock` work without a kernel restart

from per3clock import config  # noqa: E402

config.set_seeds()
config.ensure_dirs()
print("per3clock root:", config.ROOT, "| Colab:", IN_COLAB)

## Minimum evidence (lecturer's brief)
*Brief, stage 5: "Apply the methods that directly answer your question." Minimum evidence: "Results table, key figures, statistical evidence" (example: "with FDR correction").*

- [ ] Methods that directly answer objectives 1–3 (per-subject peak times; 4/4 vs 5/5 comparison; melatonin-relative phase)
- [ ] **Results table** (estimates, effect sizes in hours with 95% CIs, test statistics, p-values)
- [ ] **Key figures** (e.g. acrophase rose plots, forest plot of shifts)
- [ ] **Statistical evidence** for each objective, with assumptions checked
- [ ] **FDR correction** (Benjamini–Hochberg) wherever many tests are run: the genome-wide rhythmicity scan, and across the primary + secondary genes
- [ ] Sensitivity analyses: `strict=True` sample set; Sleep Restriction; probe-mean (`collapse="mean"`) vs selected probe
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty. With ~12 subjects per genotype, a non-significant shift is *inconclusive*, not proof of no shift. Give the CI.

## Contract
**Sample sets (shared defaults, `data/README.md`):**
- objectives 1–2 (clock time): `data.analysis_samples()` → 24 subjects (11 × 4/4, 13 × 5/5), 221 samples
- objective 3 (melatonin phase, `time_col="circadian_phase_h"`): `data.analysis_samples(for_phase_analysis=True)` → 23 subjects (AF0182 SE dropped)
- sensitivity: `strict=True` (20 subjects) and `condition="Sleep Restriction"`

**Inputs:** `data.load_clock_genes()` (one probe per gene; `collapse="mean"` for the probe sensitivity check), expression matrix via `data.collapse_probes_to_genes(data.load_expression_matrix())`.

**Implement in `per3clock.rhythm`:** `fit_subject_cosinors`, `rhythmicity_scan` (`cosinor_fit` exists: extend and validate it). **In `per3clock.circstats`:** `circular_mean_h`, `circular_diff_h`, `watson_williams_test`, `permutation_test_acrophase`, `bootstrap_ci_diff_h`.

**Outputs:**
- `results/tables/acrophase_by_subject.csv`: subject × gene × condition × time base (clock time / circadian phase)
- `results/tables/acrophase_genotype_tests.csv`: gene, time base, condition, sample set, mean acrophase 4/4 and 5/5, shift (h, 5/5 − 4/4), 95% CI, permutation p, Watson–Williams p, BH q
- `results/tables/rhythmic_genes_SE.csv`: genome-wide scan with q-values (**input to Part 4**)
- figures: `acrophase_rose_PER2.png` / `_PER3` / `_NR1D1`, `acrophase_shift_forest.png`

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data, plotting

plotting.set_style()

## 5.1 Per-subject cosinor fits (objective 1)

In [ ]:
# Load the agreed primary analysis sample (Sleep Extension)
analysis_meta = data.analysis_samples()

# Load the expression values for the selected clock genes
clock_long = data.load_clock_genes()

# Keep only samples included in the primary analysis
selected_sample_ids = analysis_meta["sample_id"].unique()

clock_long = clock_long[
    clock_long["sample_id"].isin(selected_sample_ids)
].copy()

# Check that the expected genes and participants are present
print("Participants:", analysis_meta["subject"].nunique())
print("Samples:", analysis_meta["sample_id"].nunique())
print("Genes:", sorted(clock_long["gene"].unique()))

display(clock_long.head())

**5.1.2: Fit the cosinor models**
Now we'll fit each participant's expression pattern twice:
- Clock time: Uses the actual time of day when each sample was collected.
- Circadian phase: Uses the participant's internal biological timing, relative to their circadian phase measurement.

In [ ]:
from per3clock import rhythm

# Focus on the three genes in our research objectives
genes_of_interest = ["PER2", "PER3", "NR1D1"]

primary_clock = clock_long[
    clock_long["gene"].isin(genes_of_interest)
].copy()

# Fit one cosinor model per participant and gene using clock time
clock_fits = rhythm.fit_subject_cosinors(
    primary_clock,
    time_col="clock_time_h",
    genes=genes_of_interest,
    period=24.0,
    min_points=4
)

# Fit the same models using circadian phase
phase_fits = rhythm.fit_subject_cosinors(
    primary_clock,
    time_col="circadian_phase_h",
    genes=genes_of_interest,
    period=24.0,
    min_points=4
)

# Label the time basis used by each model
clock_fits["time_base"] = "clock_time"
phase_fits["time_base"] = "circadian_phase"

# Combine both sets of results
acrophase_by_subject = pd.concat(
    [clock_fits, phase_fits],
    ignore_index=True
)

# Check the results
print("Total cosinor fits:", len(acrophase_by_subject))
print("Participants:", acrophase_by_subject["subject"].nunique())
print("Genes:", sorted(acrophase_by_subject["gene"].unique()))
print("\nFits by gene and time basis:")

display(
    acrophase_by_subject.groupby(
        ["gene", "time_base"]
    ).size().rename("number_of_fits").reset_index()
)

display(acrophase_by_subject.head())

**5.1.3: Save partcipant level results**

In [ ]:
from pathlib import Path

# Recreate the clock-time fits using the primary 24-participant sample
clock_fits = rhythm.fit_subject_cosinors(
    primary_clock,
    time_col="clock_time_h",
    genes=genes_of_interest,
    period=24.0,
    min_points=4
)

# Circadian-phase analysis uses the separate 23-participant sample
phase_meta = data.analysis_samples(for_phase_analysis=True)
phase_sample_ids = phase_meta["sample_id"].unique()

phase_long = data.load_clock_genes()
phase_long = phase_long[
    (phase_long["sample_id"].isin(phase_sample_ids)) &
    (phase_long["gene"].isin(genes_of_interest))
].copy()

phase_fits = rhythm.fit_subject_cosinors(
    phase_long,
    time_col="circadian_phase_h",
    genes=genes_of_interest,
    period=24.0,
    min_points=4
)

# Label and combine the two time bases
clock_fits["time_base"] = "clock_time"
phase_fits["time_base"] = "circadian_phase"

acrophase_by_subject = pd.concat(
    [clock_fits, phase_fits],
    ignore_index=True
)

# Save the results
output_dir = Path("results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "acrophase_by_subject.csv"
acrophase_by_subject.to_csv(output_file, index=False)

# Verify the saved analysis
print("Saved to:", output_file)
print("Total fits:", len(acrophase_by_subject))
print("\nFits by time basis and gene:")

display(
    acrophase_by_subject.groupby(
        ["time_base", "gene"]
    ).size().rename("number_of_fits").reset_index()
)

## 5.2 Genotype comparison in clock time (objective 2)

In [ ]:
from per3clock import circstats
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd

# Store the comparison results
comparison_results = []

# Analyse each gene separately under each time basis
for (gene, time_base), group in acrophase_by_subject.groupby(
    ["gene", "time_base"]
):

    # Separate participants according to genotype
    group_44 = group.loc[
        group["per3_genotype"] == "4/4", "acrophase_h"
    ].dropna().to_numpy()

    group_55 = group.loc[
        group["per3_genotype"] == "5/5", "acrophase_h"
    ].dropna().to_numpy()

    # Remove any non-finite values
    group_44 = group_44[np.isfinite(group_44)]
    group_55 = group_55[np.isfinite(group_55)]

    # Calculate circular mean peak times
    mean_44 = circstats.circular_mean_h(group_44)
    mean_55 = circstats.circular_mean_h(group_55)

    # Calculate the genotype shift: 5/5 minus 4/4
    shift = circstats.circular_diff_h(mean_55, mean_44)

    # Permutation test
    permutation = circstats.permutation_test_acrophase(
        group_55,
        group_44,
        n_perm=10_000
    )

    # Bootstrap 95% confidence interval
    ci_lower, ci_upper = circstats.bootstrap_ci_diff_h(
        group_55,
        group_44,
        n_boot=10_000,
        alpha=0.05
    )

    # Watson-Williams test
    ww = circstats.watson_williams_test(
        group_44,
        group_55
    )

    comparison_results.append({
        "gene": gene,
        "time_base": time_base,
        "condition": "Sleep Extension",
        "sample_set": "Primary",
        "n_44": len(group_44),
        "n_55": len(group_55),
        "mean_acrophase_44": mean_44,
        "mean_acrophase_55": mean_55,
        "shift_h": shift,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "permutation_p": permutation["p"],
        "watson_williams_p": ww["p"]
    })

# Create the results table
genotype_tests = pd.DataFrame(comparison_results)

# Apply Benjamini-Hochberg correction across all six comparisons
genotype_tests["permutation_q"] = multipletests(
    genotype_tests["permutation_p"],
    method="fdr_bh"
)[1]

genotype_tests["watson_williams_q"] = multipletests(
    genotype_tests["watson_williams_p"],
    method="fdr_bh"
)[1]

# Display the results
display(
    genotype_tests.round(4)
)

## 5.3 Melatonin-relative phase (objective 3)

In [ ]:
# Extract only the circadian-phase comparisons
phase_results = genotype_tests[
    genotype_tests["time_base"] == "circadian_phase"
].copy()

# Display the results
display(
    phase_results[
        [
            "gene",
            "n_44",
            "n_55",
            "mean_acrophase_44",
            "mean_acrophase_55",
            "shift_h",
            "ci_lower",
            "ci_upper",
            "permutation_p",
            "permutation_q"
        ]
    ].round(4)
)

Save comparison results

In [ ]:
from pathlib import Path

output_dir = Path("results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

phase_results.to_csv(
    output_dir / "acrophase_circadian_phase_tests.csv",
    index=False
)

print("Circadian-phase comparison results saved.")
display(phase_results)

## 5.4 Sensitivity: Sleep Restriction

Prepare the sensitivity dataset

In [ ]:
# Prepare the three sensitivity-analysis datasets

sensitivity_datasets = {}

# 1. Strict Sleep Extension sample (20 participants)
strict_meta = data.analysis_samples(strict=True)
strict_ids = strict_meta["sample_id"].unique()

strict_long = data.load_clock_genes()
strict_long = strict_long[
    (strict_long["sample_id"].isin(strict_ids)) &
    (strict_long["gene"].isin(genes_of_interest))
].copy()

sensitivity_datasets["Strict_SE"] = strict_long


# 2. Sleep Restriction sample (24 participants)
sr_meta = data.analysis_samples(condition="Sleep Restriction")
sr_ids = sr_meta["sample_id"].unique()

sr_long = data.load_clock_genes()
sr_long = sr_long[
    (sr_long["sample_id"].isin(sr_ids)) &
    (sr_long["gene"].isin(genes_of_interest))
].copy()

sensitivity_datasets["Sleep_Restriction"] = sr_long


# 3. Probe-mean Sleep Extension sample
mean_long = data.load_clock_genes(collapse="mean")
mean_long = mean_long[
    (mean_long["sample_id"].isin(selected_sample_ids)) &
    (mean_long["gene"].isin(genes_of_interest))
].copy()

sensitivity_datasets["Probe_Mean_SE"] = mean_long


# Check the datasets
for name, dataset in sensitivity_datasets.items():
    print(f"\n{name}")
    print("Participants:", dataset["subject"].nunique())
    print("Samples:", dataset["sample_id"].nunique())
    print("Genes:", sorted(dataset["gene"].unique()))

Fit the sensitivity to the cosinor models

In [ ]:
sensitivity_fits = []

for sample_set, dataset in sensitivity_datasets.items():

    fits = rhythm.fit_subject_cosinors(
        dataset,
        time_col="clock_time_h",
        genes=genes_of_interest,
        period=24.0,
        min_points=4
    )

    fits["time_base"] = "clock_time"
    fits["sample_set"] = sample_set

    sensitivity_fits.append(fits)

# Combine all sensitivity results
sensitivity_acrophases = pd.concat(
    sensitivity_fits,
    ignore_index=True
)

# Check the number of fits
print("Total sensitivity fits:", len(sensitivity_acrophases))

display(
    sensitivity_acrophases.groupby(
        ["sample_set", "gene"]
    ).size().rename("number_of_fits").reset_index()
)

Compare the genotype peak times across sensitivity analyses

In [ ]:
sensitivity_comparisons = []

for sample_set, dataset in sensitivity_acrophases.groupby("sample_set"):

    for gene, group in dataset.groupby("gene"):

        # Separate the two genotype groups
        group_44 = group.loc[
            group["per3_genotype"] == "4/4", "acrophase_h"
        ].dropna().to_numpy()

        group_55 = group.loc[
            group["per3_genotype"] == "5/5", "acrophase_h"
        ].dropna().to_numpy()

        group_44 = group_44[np.isfinite(group_44)]
        group_55 = group_55[np.isfinite(group_55)]

        # Circular mean peak times
        mean_44 = circstats.circular_mean_h(group_44)
        mean_55 = circstats.circular_mean_h(group_55)

        # Difference: 5/5 minus 4/4
        shift = circstats.circular_diff_h(mean_55, mean_44)

        # Statistical tests
        permutation = circstats.permutation_test_acrophase(
            group_55, group_44, n_perm=10_000
        )

        ci_lower, ci_upper = circstats.bootstrap_ci_diff_h(
            group_55, group_44, n_boot=10_000
        )

        ww = circstats.watson_williams_test(
            group_44, group_55
        )

        sensitivity_comparisons.append({
            "gene": gene,
            "time_base": "clock_time",
            "condition": (
                "Sleep Restriction"
                if sample_set == "Sleep_Restriction"
                else "Sleep Extension"
            ),
            "sample_set": sample_set,
            "n_44": len(group_44),
            "n_55": len(group_55),
            "mean_acrophase_44": mean_44,
            "mean_acrophase_55": mean_55,
            "shift_h": shift,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
            "permutation_p": permutation["p"],
            "watson_williams_p": ww["p"]
        })

# Create the sensitivity comparison table
sensitivity_tests = pd.DataFrame(sensitivity_comparisons)

# Apply FDR correction separately within each sensitivity dataset
for sample_set in sensitivity_tests["sample_set"].unique():

    mask = sensitivity_tests["sample_set"] == sample_set

    sensitivity_tests.loc[mask, "permutation_q"] = multipletests(
        sensitivity_tests.loc[mask, "permutation_p"],
        method="fdr_bh"
    )[1]

    sensitivity_tests.loc[mask, "watson_williams_q"] = multipletests(
        sensitivity_tests.loc[mask, "watson_williams_p"],
        method="fdr_bh"
    )[1]

# Display results
display(
    sensitivity_tests.sort_values(
        ["sample_set", "gene"]
    ).round(4)
)

Circadian-pahse sensitivity fits

In [ ]:
# Prepare phase-analysis datasets using the shared selection rules

phase_sensitivity_settings = {
    "Strict_SE": {
        "condition": "Sleep Extension",
        "strict": True,
        "collapse": "max_mean"
    },
    "Sleep_Restriction": {
        "condition": "Sleep Restriction",
        "strict": False,
        "collapse": "max_mean"
    },
    "Probe_Mean_SE": {
        "condition": "Sleep Extension",
        "strict": False,
        "collapse": "mean"
    }
}

phase_sensitivity_fits = []

for sample_set, settings in phase_sensitivity_settings.items():

    # Select participants with valid circadian-phase measurements
    phase_meta = data.analysis_samples(
        condition=settings["condition"],
        strict=settings["strict"],
        for_phase_analysis=True
    )

    # Load expression using the specified probe-selection method
    phase_data = data.load_clock_genes(
        collapse=settings["collapse"]
    )

    # Keep only the selected samples and research genes
    phase_ids = phase_meta["sample_id"].unique()

    phase_data = phase_data[
        (phase_data["sample_id"].isin(phase_ids)) &
        (phase_data["gene"].isin(genes_of_interest))
    ].copy()

    # Fit one model per participant and gene using circadian phase
    fits = rhythm.fit_subject_cosinors(
        phase_data,
        time_col="circadian_phase_h",
        genes=genes_of_interest,
        period=24.0,
        min_points=4
    )

    fits["time_base"] = "circadian_phase"
    fits["sample_set"] = sample_set

    phase_sensitivity_fits.append(fits)

    print(
        f"{sample_set}: "
        f"{phase_meta['subject'].nunique()} participants, "
        f"{len(fits)} fits"
    )

# Combine all phase sensitivity fits
phase_sensitivity_acrophases = pd.concat(
    phase_sensitivity_fits,
    ignore_index=True
)

print("\nTotal phase sensitivity fits:",
      len(phase_sensitivity_acrophases))

display(
    phase_sensitivity_acrophases.groupby(
        ["sample_set", "gene"]
    ).size().rename("number_of_fits").reset_index()
)

Compare the genotypes again with the circadian phase

In [ ]:
phase_sensitivity_comparisons = []

for sample_set, dataset in phase_sensitivity_acrophases.groupby("sample_set"):

    for gene, group in dataset.groupby("gene"):

        group_44 = group.loc[
            group["per3_genotype"] == "4/4", "acrophase_h"
        ].dropna().to_numpy()

        group_55 = group.loc[
            group["per3_genotype"] == "5/5", "acrophase_h"
        ].dropna().to_numpy()

        group_44 = group_44[np.isfinite(group_44)]
        group_55 = group_55[np.isfinite(group_55)]

        # Circular mean peak times
        mean_44 = circstats.circular_mean_h(group_44)
        mean_55 = circstats.circular_mean_h(group_55)

        # Difference: 5/5 minus 4/4
        shift = circstats.circular_diff_h(mean_55, mean_44)

        # Permutation test
        permutation = circstats.permutation_test_acrophase(
            group_55, group_44, n_perm=10_000
        )

        # Bootstrap confidence interval
        ci_lower, ci_upper = circstats.bootstrap_ci_diff_h(
            group_55, group_44, n_boot=10_000
        )

        # Watson-Williams test
        ww = circstats.watson_williams_test(
            group_44, group_55
        )

        phase_sensitivity_comparisons.append({
            "gene": gene,
            "time_base": "circadian_phase",
            "condition": (
                "Sleep Restriction"
                if sample_set == "Sleep_Restriction"
                else "Sleep Extension"
            ),
            "sample_set": sample_set,
            "n_44": len(group_44),
            "n_55": len(group_55),
            "mean_acrophase_44": mean_44,
            "mean_acrophase_55": mean_55,
            "shift_h": shift,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
            "permutation_p": permutation["p"],
            "watson_williams_p": ww["p"]
        })

phase_sensitivity_tests = pd.DataFrame(
    phase_sensitivity_comparisons
)

# Apply BH correction across all nine phase comparisons
phase_sensitivity_tests["permutation_q"] = multipletests(
    phase_sensitivity_tests["permutation_p"],
    method="fdr_bh"
)[1]

phase_sensitivity_tests["watson_williams_q"] = multipletests(
    phase_sensitivity_tests["watson_williams_p"],
    method="fdr_bh"
)[1]

display(
    phase_sensitivity_tests.sort_values(
        ["sample_set", "gene"]
    ).round(4)
)

Save the results

In [ ]:
output_dir = Path("results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

phase_sensitivity_acrophases.to_csv(
    output_dir / "acrophase_phase_sensitivity_by_subject.csv",
    index=False
)

phase_sensitivity_tests.to_csv(
    output_dir / "acrophase_phase_sensitivity_tests.csv",
    index=False
)

print("Circadian-phase sensitivity results saved.")

## 5.5 Genome-wide rhythmicity scan

In [ ]:
# Select the main Sleep Extension samples
genome_meta = data.analysis_samples()

# Load the full expression matrix
expression_matrix = data.load_expression_matrix()

# Collapse probes into genes using the selected-probe method
gene_expression = data.collapse_probes_to_genes(
    expression_matrix,
    method="max_mean"
)

# Keep only samples included in the main analysis
selected_ids = genome_meta["sample_id"].tolist()

gene_expression = gene_expression[
    [sample for sample in selected_ids if sample in gene_expression.columns]
]

# Align metadata with the expression matrix
genome_meta = genome_meta[
    genome_meta["sample_id"].isin(gene_expression.columns)
].copy()

print("Participants:", genome_meta["subject"].nunique())
print("Samples:", len(genome_meta))
print("Genes:", len(gene_expression))
print("Expression matrix shape:", gene_expression.shape)

Fit individual participant-level cosinor models

In [ ]:
genome_results = []

for subject, subject_meta in genome_meta.groupby("subject"):

    subject_samples = subject_meta["sample_id"].tolist()

    # Get the circadian phase for this participant's samples
    times = subject_meta.set_index("sample_id").loc[
        subject_samples, "circadian_phase_h"
    ].to_numpy()

    # Get this participant's expression measurements
    subject_expression = gene_expression[subject_samples]

    for gene, expression in subject_expression.iterrows():

        try:
            fit = rhythm.cosinor_fit(
                times,
                expression.to_numpy(),
                period=24.0
            )

            genome_results.append({
                "subject": subject,
                "gene": gene,
                "per3_genotype": subject_meta["per3_genotype"].iloc[0],
                "amplitude": fit["amplitude"],
                "acrophase_h": fit["acrophase_h"],
                "r2": fit["r2"],
                "p": fit["p"],
                "n": fit["n"]
            })

        except ValueError:
            continue

genome_subject_fits = pd.DataFrame(genome_results)

print("Participant-level fits:", len(genome_subject_fits))
print("Unique genes:", genome_subject_fits["gene"].nunique())

display(genome_subject_fits.head())

Combine participant level p values

In [ ]:
from scipy.stats import combine_pvalues
from statsmodels.stats.multitest import multipletests

gene_rhythm_results = []

for gene, group in genome_subject_fits.groupby("gene"):

    p_values = group["p"].dropna().to_numpy()

    if len(p_values) == 0:
        continue

    # Combine participant-level p-values
    combined_stat, combined_p = combine_pvalues(
        p_values,
        method="fisher"
    )

    gene_rhythm_results.append({
        "gene": gene,
        "participants": len(p_values),
        "fisher_statistic": combined_stat,
        "p": combined_p,
        "mean_amplitude": group["amplitude"].mean(),
        "mean_r2": group["r2"].mean()
    })

genome_rhythm_scan = pd.DataFrame(gene_rhythm_results)

# Correct for multiple testing across all genes
genome_rhythm_scan["q"] = multipletests(
    genome_rhythm_scan["p"],
    method="fdr_bh"
)[1]

# Sort genes by adjusted p-value
genome_rhythm_scan = genome_rhythm_scan.sort_values("q")

display(genome_rhythm_scan.head(10))

Compare our target genes

In [ ]:
target_genes = ["PER2", "PER3", "NR1D1"]

target_results = genome_rhythm_scan[
    genome_rhythm_scan["gene"].isin(target_genes)
].copy()

display(target_results)

Save the results

In [ ]:
from pathlib import Path

output_path = Path("results/tables/rhythmic_genes_SE.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

genome_rhythm_scan.to_csv(output_path)

print("Saved:", output_path)

## Interpretation

The analysis found that PER2, PER3 and NR1D1 all showed significant evidence of rhythmic expression across participants.

However, the genotype comparisons did not establish that PER3 5/5 carriers consistently reached peak expression earlier than 4/4 carriers.
- Clock time: NR1D1 peaked approximately 1.16 hours earlier and PER2 approximately 0.51 hours earlier in 5/5 carriers. However, both differences were statistically non-significant.
- Circadian phase: The earlier timing was not maintained relative to melatonin phase. The estimated shifts varied between the three genes, with no statistically significant differences.
- Sensitivity analyses: Results were inconsistent across stricter participant selection, Sleep Restriction and probe-mean analysis. None established a significant genotype effect.
Overall conclusion: The findings do not provide sufficient evidence that PER3 genotype influences the peak-expression timing of the three clock genes. The results are inconclusive rather than proof that no relationship exists, partly because of the small sample size.